# 🎤 OmniVoice V7 — Your Voice, Your Script
**আপনার নিজের লেখা স্ক্রিপ্ট, মেল ভয়েসে, একটি ফাইলে।**

Run all cells sequentially in Kaggle with **GPU T4** enabled.

**শুধু Step 4-এ `MY_SCRIPT` ভ্যারিয়েবলে আপনার টেক্সট পেস্ট করুন।**

In [ ]:
# ⚙️ Step 1: Check GPU Status
!nvidia-smi
import torch
print(f"\n✅ CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 📦 Step 2: Clone & Install OmniVoice Studio
!pip install -q uv
!git clone https://github.com/debpalash/OmniVoice-Studio.git /kaggle/working/omnivoice-studio
%cd /kaggle/working/omnivoice-studio
!uv sync

In [ ]:
# ⚡ Step 3: Launch OmniVoice Backend Server in Background
import os, time, subprocess
%cd /kaggle/working/omnivoice-studio

log_file = open("/tmp/omnivoice.log", "w")
subprocess.Popen(["uv", "run", "python", "backend/main.py"], stdout=log_file, stderr=log_file)

print("⏳ Waiting 15s for server startup...")
time.sleep(15)
!curl -sf http://localhost:3900/health || echo '❌ Server starting failed! Check /tmp/omnivoice.log'

In [ ]:
# =============================================================================
# 🎤 Step 4: V7 — YOUR VOICE, YOUR SCRIPT
# =============================================================================

import os, time, json, urllib.request

BASE_OUTPUT_DIR = "/kaggle/working/outputs/v7_custom_prompt"
API_URL = "http://localhost:3900/v1/audio/speech"

def gen(text, voice="onyx", model="tts-1-hd", output_path="",
        speed=1.0, num_step=32, guidance_scale=2.0):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    payload = {
        "model": model, "voice": voice, "input": text,
        "response_format": "wav", "speed": speed,
        "num_step": num_step, "guidance_scale": guidance_scale,
    }
    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(API_URL, data=data,
                                headers={"Content-Type": "application/json"})
    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=600) as resp:
            content = resp.read()
            with open(output_path, "wb") as f:
                f.write(content)
            dur = time.time() - t0
            kb = len(content) // 1024
            print(f"  ✅ DONE!")
            print(f"     📁 {os.path.basename(output_path)}")
            print(f"     📊 {kb} KB")
            print(f"     ⏱️  Generation time: {dur:.1f}s")
    except Exception as e:
        print(f"  ❌ FAILED: {e}")

# =============================================================================
# ✏️ আপনার স্ক্রিপ্ট এখানে লিখুন ↓↓↓
# =============================================================================

MY_SCRIPT = \"\"\"
PUT YOUR SCRIPT HERE. REPLACE THIS ENTIRE TEXT WITH YOUR OWN.
\"\"\"

# =============================================================================
# ⚙️ SETTINGS
# =============================================================================
VOICE = "onyx"          # মেল ভয়েস: onyx (গভীর), echo (মধ্যম), alloy (হালকা)
SPEED = 0.88            # 0.7=খুব ধীর, 0.88=সিনেমাটিক, 1.0=স্বাভাবিক
GUIDANCE = 2.5          # 1.5=শান্ত, 2.0=স্বাভাবিক, 2.5=আবেগপূর্ণ, 3.0=চরম
FILENAME = "v7_custom"  # আউটপুট ফাইলের নাম

# =============================================================================
# 🚀 GENERATE
# =============================================================================
text = MY_SCRIPT.strip()

print("=" * 70)
print("🎤  VERSION 7 — YOUR VOICE, YOUR SCRIPT")
print("=" * 70)
print(f"📝  Script: {len(text)} chars")
print(f"🎙️  Voice: {VOICE}")
print(f"⚡  Speed: {SPEED} | Guidance: {GUIDANCE}")
print("=" * 70)

print(f"\n🎤 Generating with {VOICE} voice...")
gen(
    text=text,
    voice=VOICE,
    model="tts-1-hd",
    output_path=f"{BASE_OUTPUT_DIR}/{FILENAME}_{VOICE}.wav",
    speed=SPEED,
    num_step=32,
    guidance_scale=GUIDANCE,
)
print(f"\n🎉  GENERATED: {FILENAME}_{VOICE}.wav")

In [ ]:
# 📦 Step 5: Zip output for Download
!apt-get install -y zip
!zip -r /kaggle/working/omnivoice_v7_custom.zip /kaggle/working/outputs/v7_custom_prompt
print("\n✅ DOWNLOAD READY!")
print("📁 Kaggle Output Tab -> Download 'omnivoice_v7_custom.zip'")